# AI-Generated Image Detector - Experimentation Notebook

This notebook provides interactive experimentation for:
1. Loading and verifying `config.yaml` settings.
2. Inspecting data augmentation transforms.
3. Instantiating and testing model backbones (ResNet, EfficientNet).
4. Running forward passes with synthetic tensors.
5. Programmatic inference with `DetectorPredictor`.

In [ ]:
import sys
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# Ensure project root is in sys.path
project_root = os.path.abspath("..")
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.utils import load_config, get_device, set_seed
from src.model import build_model, AIImageDetector
from src.data_loader import get_transforms

set_seed(42)
print(f"PyTorch Version: {torch.__version__}")
device = get_device("auto")
print(f"Active Device: {device}")

## 1. Load Configuration

In [ ]:
config = load_config("../config.yaml")
print("Configuration loaded successfully:")
for key, val in config.items():
    print(f"{key.upper()}: {val}")

## 2. Inspect Data Augmentation Pipeline

In [ ]:
# Get train augmentation transform
train_transform = get_transforms(config, split="train")
valid_transform = get_transforms(config, split="valid")

print("Train Transform Pipeline:")
print(train_transform)

print("\nValidation / Test Pipeline:")
print(valid_transform)

In [ ]:
# Create a synthetic test image to visualize augmentations
dummy_img = Image.new("RGB", (256, 256), color=(73, 109, 137))
# Add some simple pattern
import ImageDraw if 'ImageDraw' in locals() else None
from PIL import ImageDraw
draw = ImageDraw.Draw(dummy_img)
draw.rectangle([50, 50, 200, 200], fill=(255, 180, 50))
draw.line([0, 0, 256, 256], fill=(255, 255, 255), width=4)

fig, axes = plt.subplots(1, 4, figsize=(12, 3))
axes[0].imshow(dummy_img)
axes[0].set_title("Original")
axes[0].axis("off")

for i in range(1, 4):
    # Apply train augmentation
    aug_tensor = train_transform(dummy_img)
    # Denormalize for display
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    aug_np = aug_tensor.numpy().transpose((1, 2, 0))
    aug_np = std * aug_np + mean
    aug_np = np.clip(aug_np, 0, 1)
    axes[i].imshow(aug_np)
    axes[i].set_title(f"Augmentation #{i}")
    axes[i].axis("off")

plt.tight_layout()
plt.show()

## 3. Instantiate and Test Model Backbones

In [ ]:
# Test ResNet50
model = build_model(config)
model.eval()

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model Backbone: {config['model']['backbone']}")
print(f"Total Parameters: {total_params:,}")
print(f"Trainable Parameters: {trainable_params:,}")

# Forward pass test with synthetic batch
dummy_batch = torch.randn(2, 3, 224, 224)
with torch.no_grad():
    output_logits = model(dummy_batch)
    probabilities = torch.softmax(output_logits, dim=1)

print(f"Output Logits Shape: {output_logits.shape}")
print(f"Output Probabilities:\n{probabilities.numpy()}")

## 4. Testing Alternative Backbones (e.g. EfficientNet)

In [ ]:
eff_model = AIImageDetector(backbone_name="efficientnet_b0", pretrained=False, num_classes=2)
eff_model.eval()
with torch.no_grad():
    eff_out = eff_model(dummy_batch)
print(f"EfficientNet-B0 Output Shape: {eff_out.shape}")

## 5. Inference Testing with DetectorPredictor

Once you have trained the model using `python src/train.py`, you can test single image inference directly:

In [ ]:
from src.predict import DetectorPredictor

weights_path = "../models/best_model.pth"
if os.path.exists(weights_path):
    predictor = DetectorPredictor(config=config, weights_path=weights_path)
    # result = predictor.predict("path/to/test_image.jpg")
    # print(result)
else:
    print(f"[INFO] Model weights not found at '{weights_path}'.")
    print("Run 'python src/train.py' after populating database folders to create a checkpoint.")